# Topic Modelling Starter

This notebook sets up a reusable workflow for LDA and BERTopic on the news dataset.

In [5]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from bertopic import BERTopic

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer



In [10]:
import nltk

nltk.download("punkt")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\WEWL\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\WEWL\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [8]:
DATA_PATH = Path("Data/NLP_data.csv")
TEXT_COLUMN = "content"
SAMPLE_SIZE = 5000

df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=[TEXT_COLUMN]).copy()
df = df[["article_id", "title", "category", TEXT_COLUMN]].copy()

if SAMPLE_SIZE and len(df) > SAMPLE_SIZE:
    df = df.sample(SAMPLE_SIZE, random_state=42).reset_index(drop=True)

df.head()

,article_id,title,category,content
0,63642,Young teen wins top science prize for soap tha...,Ethiopia,"A 14-year-old boy has been named ""America's to..."
1,13370,"Vehicular homicide suspect who ""reeked of alco...",United States,"Ting Ye, 26, ""reeked of alcohol"" when she was ..."
2,69158,The Pastry Chefs Defining Restaurant Dessert R...,Guyana,Kelly Nam plates a dessert at New York City’s ...
3,97723,"Flula Borg hilarious deconstructs that weird ""...",History,"Ok, one last Halloween post and I'll abandon t..."
4,84216,Korat zoo welcomes new member - sun bear,Myanmar,Nakhon Ratchasima Zoo has a new member - a one...


In [11]:
stop_words = set(stopwords.words("english"))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = text.split()
    tokens = [token for token in tokens if len(token) > 2 and token not in stop_words]
    return tokens

df["tokens"] = df[TEXT_COLUMN].apply(clean_text)
df["clean_text"] = df["tokens"].apply(lambda tokens: " ".join(tokens))

df[["title", "category", "clean_text"]].head()

,title,category,clean_text
0,Young teen wins top science prize for soap tha...,Ethiopia,year old boy named america top young scientist...
1,"Vehicular homicide suspect who ""reeked of alco...",United States,ting reeked alcohol pulled driver seat mph rol...
2,The Pastry Chefs Defining Restaurant Dessert R...,Guyana,kelly nam plates dessert new york city joomak ...
3,"Flula Borg hilarious deconstructs that weird ""...",History,one last halloween post abandon topic next yea...
4,Korat zoo welcomes new member - sun bear,Myanmar,nakhon ratchasima zoo new member one month old...


In [12]:
print(df.shape)
df["category"].fillna("Unknown").value_counts().head(10)

(5000, 6)


category
Stock          180
Technology     120
Finance        116
Real estate    115
Health         109
Canada         107
Education       97
News            93
Food            90
Jobs            86
Name: count, dtype: int64

In [13]:
N_TOPICS = 8

vectorizer = CountVectorizer(max_df=0.95, min_df=10, stop_words="english")
doc_term_matrix = vectorizer.fit_transform(df["clean_text"])

lda_model = LatentDirichletAllocation(
    n_components=N_TOPICS,
    random_state=42,
    learning_method="batch"
)
lda_model.fit(doc_term_matrix)

,n_components,8
,doc_topic_prior,None
,topic_word_prior,None
,learning_method,'batch'
,learning_decay,0.7
,learning_offset,10.0
,max_iter,10
,batch_size,128
,evaluate_every,-1
,total_samples,1000000.0
,perp_tol,0.1


In [14]:
def print_lda_topics(model, feature_names, top_n=10):
    for topic_idx, topic in enumerate(model.components_):
        top_words = [feature_names[i] for i in topic.argsort()[:-top_n - 1:-1]]
        print(f"Topic {topic_idx + 1}: {', '.join(top_words)}")

feature_names = vectorizer.get_feature_names_out()
print_lda_topics(lda_model, feature_names)

Topic 1: report, free, quarter, recent, according, shares, nyse, company, second, filing
Topic 2: said, news, president, south, people, minister, city, new, images, nov
Topic 3: october, time, prime, year, apple, minister, news, short, day, month
Topic 4: report, globe, newswire, nov, research, market, free, global, rating, com
Topic 5: november, said, season, tuesday, october, series, thursday, united, time, record
Topic 6: company, million, new, earnings, announced, media, reported, quarter, years, group
Topic 7: world, israel, hamas, gaza, cup, war, new, india, including, getty
Topic 8: new, traded, state, high, week, york, low, thursday, trading, monday


In [15]:
lda_topic_matrix = lda_model.transform(doc_term_matrix)
df["lda_topic"] = lda_topic_matrix.argmax(axis=1)
df[["title", "category", "lda_topic"]].head(10)

,title,category,lda_topic
0,Young teen wins top science prize for soap tha...,Ethiopia,2
1,"Vehicular homicide suspect who ""reeked of alco...",United States,7
2,The Pastry Chefs Defining Restaurant Dessert R...,Guyana,7
3,"Flula Borg hilarious deconstructs that weird ""...",History,1
4,Korat zoo welcomes new member - sun bear,Myanmar,7
5,Striking actors say they have rejected the Hol...,Artificial Intelligence,3
6,Addus HomeCare (NASDAQ:ADUS) Rating Lowered to...,Health,3
7,India looks to fast-track approvals for Elon M...,Cars,2
8,Here Are the Favorites to Win the 2023 Nobel P...,Ukraine,7
9,Unity Software Inc. (NYSE:U) Shares Sold by Re...,Germany,0


In [16]:
dictionary = corpora.Dictionary(df["tokens"])
corpus = [dictionary.doc2bow(tokens) for tokens in df["tokens"]]
topic_terms = []

for topic in lda_model.components_:
    top_indices = topic.argsort()[:-11:-1]
    topic_terms.append([feature_names[i] for i in top_indices])

coherence_model = CoherenceModel(
    topics=topic_terms,
    texts=df["tokens"],
    dictionary=dictionary,
    coherence="c_v"
)

print({"lda_coherence": coherence_model.get_coherence()})

NameError: name 'corpora' is not defined

In [17]:
topic_model = BERTopic(language="english", calculate_probabilities=True, verbose=True)
bertopic_topics, bertopic_probs = topic_model.fit_transform(df["clean_text"].tolist())
df["bertopic_topic"] = bertopic_topics

2026-03-24 16:29:16,851 - BERTopic - Embedding - Transforming documents to embeddings.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

2026-03-24 16:29:47,896 - BERTopic - Embedding - Completed ✓
2026-03-24 16:29:47,897 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-24 16:30:13,587 - BERTopic - Dimensionality - Completed ✓
2026-03-24 16:30:13,588 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-24 16:30:15,087 - BERTopic - Cluster - Completed ✓
2026-03-24 16:30:15,095 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-24 16:30:15,275 - BERTopic - Representation - Completed ✓


In [ ]:
topic_info = topic_model.get_topic_info()
topic_info.head(10)

In [ ]:
topic_model.get_topic(0)

In [ ]:
# Run in Jupyter to open the interactive visualisation.\n
topic_model.visualize_barchart(top_n_topics=10)